In [1]:
# Standard header to load the Kalshi API and related modules, ensuring the current directory (or "Python" subdirectory) is in the Python path.

from pathlib import Path
import sys

python_dir = Path.cwd()
if not (python_dir / "kalshi_api.py").exists():
    python_dir = Path.cwd() / "Python"

if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

from kalshi_api import *
from kalshi_analysis import *
from kalshi_plot import *


In [2]:
def get_clean_nba_games(status="open", limit=200):
        raw_df = get_nba_games_df(status=status, limit=limit)
        if raw_df.empty:
            print(f"No NBA markets returned for status='{status}'.")
            return raw_df

        parsed_df = raw_df["Ticker"].apply(parse_kxnbagame_ticker).apply(pd.Series)
        clean_df = pd.concat([raw_df, parsed_df], axis=1)
        clean_df = clean_df.sort_values(
            ["GameDate", "Home", "Away", "Ticker"],
            na_position="last"
        ).reset_index(drop=True)
        return clean_df


In [3]:

def _nba_season_from_game_date(game_date) -> int:
    game_date = pd.to_datetime(game_date)
    return game_date.year if game_date.month >= 10 else game_date.year - 1

In [6]:

def LoadMissingNBATrades(
    season: int,
    base_path: str = r"E:\PredMktData\TradeExports"
):
    error_columns = ["Ticker", "Error"]
    trade_path = os.path.join(base_path, f"NBA{season}")
    os.makedirs(trade_path, exist_ok=True)

    settled_games = get_clean_nba_games(status="settled")
    if settled_games.empty:
        print(f"No settled NBA games returned for season {season}.")
        return pd.DataFrame(columns=error_columns)

    settled_games = settled_games.copy()
    settled_games["Season"] = settled_games["GameDate"].apply(_nba_season_from_game_date)
    df_settled = settled_games.loc[
        settled_games["Season"] == season,
        ["Ticker"]
    ]

    if df_settled.empty:
        print(f"No settled NBA games found for season {season}.")
        return pd.DataFrame(columns=error_columns)

    df_trade = GetTradeFiles(trade_path)
    df_load = df_settled[~df_settled["Ticker"].isin(df_trade["Ticker"])]

    if df_load.empty:
        print(f"No missing NBA trade files for season {season}.")
        return pd.DataFrame(columns=error_columns)

    error_log = []

    for ticker in df_load["Ticker"]:
        print(f"\nProcessing {ticker}...")
        try:
            GetUnsavedTrades(ticker, save_path=trade_path)
        except Exception as e:
            print(f"Error processing {ticker}: {e}")
            error_log.append({"Ticker": ticker, "Error": str(e)})

    return pd.DataFrame(error_log, columns=error_columns)

In [4]:
df = get_clean_nba_games(status="settled", limit=500)

In [7]:
LoadMissingNBATrades(2025)

Found 0 trade files in E:\PredMktData\TradeExports\NBA2025

Processing KXNBAGAME-26MAR27ATLBOS-ATL...


c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:375: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[yes_trades, "price"] = df.loc[yes_trades, "price"].fillna(
c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:379: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[no_trades, "price"] = df.loc[no_trades, "price"].fillna(


Saved 6626 rows to E:\PredMktData\TradeExports\NBA2025\KXNBAGAME-26MAR27ATLBOS-ATL-Trades.csv

Processing KXNBAGAME-26MAR27ATLBOS-BOS...


c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:375: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[yes_trades, "price"] = df.loc[yes_trades, "price"].fillna(
c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:379: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[no_trades, "price"] = df.loc[no_trades, "price"].fillna(


Saved 7373 rows to E:\PredMktData\TradeExports\NBA2025\KXNBAGAME-26MAR27ATLBOS-BOS-Trades.csv

Processing KXNBAGAME-26MAR27MIACLE-CLE...


...

c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:375: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[yes_trades, "price"] = df.loc[yes_trades, "price"].fillna(
c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:379: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[no_trades, "price"] = df.loc[no_trades, "price"].fillna(


Saved 1261 rows to E:\PredMktData\TradeExports\NBA2025\KXNBAGAME-26MAR27MIACLE-MIA-Trades.csv

Processing KXNBAGAME-26MAR27UTADEN-DEN...


c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:375: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[yes_trades, "price"] = df.loc[yes_trades, "price"].fillna(
c:\Users\Owner\Dropbox\ECU Misc\Active Working Papers\Prediction Markets\KalshiNFL\Python\kalshi_api.py:379: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[no_trades, "price"] = df.loc[no_trades, "price"].fillna(


Saved 13445 rows to E:\PredMktData\TradeExports\NBA2025\KXNBAGAME-26MAR27UTADEN-DEN-Trades.csv

Processing KXNBAGAME-26MAR27UTADEN-UTA...


KeyboardInterrupt: 

In [15]:
df1 = get_trades("KXNBAGAME-26MAR27ATLBOS-ATL")
df2 = get_trades("KXNFLGAME-26SEP09NESEA-NE")